# Day 7 · Lab 2 — Report Generation with Charts + Consistency Verification

## What you'll build

1. Load insights from Lab 1
2. Render a markdown memo report with inline traceability
3. Generate matplotlib charts from LLM-produced code
4. Run consistency verification — every number in prose appears in chart data
5. Progressive compression: full report → 500-word summary → 5-line bullets

## Prerequisites

- Lab 1 complete — `/tmp/day7_insights.json` exists
- matplotlib installed (pre-installed with conda base)

## Step 1 — Environment

In [ ]:
import os, sys, subprocess, json, re, io, base64
from pathlib import Path

for pkg in ["python-dotenv", "langchain-openai", "matplotlib"]:
    try: __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "": del os.environ[k]

os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

insights_data = json.loads(Path("/tmp/day7_insights.json").read_text())
insights = insights_data["insights"]
source_data = insights_data["source_data"]

print(f"✓ Loaded {len(insights)} insights from Lab 1")

## Step 2 — Draft the full memo report

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


MEMO_TEMPLATE = '''# Q4 Business Review — Draft

**Date:** _pending_
**Author:** BA Reporting Agent
**Recipient:** CFO

## Executive Overview

{overview}

## Key Insights

{insights_section}

## Recommendations

{recommendations_section}

---
_Traceability: every claim in this memo links to a source data point or requirement ID._
'''


def draft_overview(insights, source_data):
    prompt = f'''Write a 3-4 sentence executive overview for a Q4 memo. Rules:
- Preserve numbers, no adjectives without data
- Focus on the TOP surprise
- Do not list individual insights (they appear below)

Top insights: {[i['claim'] for i in insights[:3]]}
Data: {source_data}
'''
    return llm.invoke(prompt).content.strip()


def draft_insights_section(insights):
    lines = []
    for ins in insights:
        lines.append(f"### {ins['claim']}")
        lines.append(f"**Data:** {', '.join(ins['supporting_data'][:3])}")
        lines.append(f"**Surprise:** {ins['surprise_level']} · **Confidence:** {ins['confidence']:.2f}")
        if ins.get('linked_requirement_id'):
            lines.append(f"**Links to:** {ins['linked_requirement_id']}")
        lines.append("")
    return "\n".join(lines)


def draft_recommendations(insights):
    lines = []
    for i, ins in enumerate(insights[:3], 1):
        lines.append(f"{i}. {ins['recommended_action']}")
    return "\n".join(lines)


overview = draft_overview(insights, source_data)
insights_section = draft_insights_section(insights)
recommendations_section = draft_recommendations(insights)

memo = MEMO_TEMPLATE.format(
    overview=overview,
    insights_section=insights_section,
    recommendations_section=recommendations_section,
)

print(memo[:1500])
print("\n...")
print(memo[-300:])

## Step 3 — Generate chart code with LLM

The LLM writes matplotlib code. We execute it. Deterministic pixels.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt


def generate_chart(insight, data_slice):
    prompt = f'''Write Python matplotlib code that plots this data to reinforce this insight.

Insight: {insight['claim']}
Data available (named 'data' in scope): {json.dumps(data_slice)}

Rules:
- Use only 'plt' and 'data' — no other imports
- Pick the RIGHT chart type (bar, line, pie, etc.)
- Add title matching the insight
- Use plt.tight_layout()
- Do NOT call plt.show() or plt.savefig()

Return ONLY the code, no fences, no explanation.'''
    raw = llm.invoke(prompt).content.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        if raw.startswith("python"):
            raw = raw[6:].strip()
    return raw


# Generate a chart for the top insight
top_insight = insights[0]
# Give the LLM the region-level data for this chart
data_slice = {
    "revenue_by_region": source_data["revenue_by_region"],
    "revenue_prior_year_q4": source_data["revenue_prior_year_q4"],
}

chart_code = generate_chart(top_insight, data_slice)
print("Generated chart code:")
print("─" * 60)
print(chart_code)

## Step 4 — Execute chart code + save PNG

In [ ]:
def execute_and_save(code_str, data_dict, out_path):
    plt.close('all')   # start fresh
    ns = {'plt': plt, 'data': data_dict}
    try:
        exec(code_str, ns)
        plt.savefig(out_path, dpi=100, bbox_inches='tight')
        plt.close('all')
        return {"ok": True, "path": str(out_path)}
    except Exception as e:
        plt.close('all')
        return {"ok": False, "error": str(e)}


chart_path = Path("/tmp/day7_top_chart.png")
result = execute_and_save(chart_code, data_slice, chart_path)

if result["ok"]:
    print(f"✓ Chart saved to {chart_path} ({chart_path.stat().st_size} bytes)")
else:
    print(f"✗ Chart execution failed: {result['error']}")

# Display inline in Jupyter
from IPython.display import Image, display
if chart_path.exists():
    display(Image(str(chart_path)))

## Step 5 — Consistency verification

Every number in prose must trace to something in the underlying data.

In [ ]:
def extract_numbers(text):
    """Extract all numeric-looking tokens from text."""
    # Match: 12,345 or 12.5% or 3.14 or 1_000_000
    pattern = r'\d[\d,._]*\d*(?:\.\d+)?%?'
    return set(re.findall(pattern, text))


def flatten_source_numbers(data):
    nums = set()
    def walk(x):
        if isinstance(x, dict):
            for v in x.values(): walk(v)
        elif isinstance(x, (list, tuple)):
            for v in x: walk(v)
        elif isinstance(x, (int, float)):
            nums.add(str(x))
            if isinstance(x, int) and x >= 1000:
                nums.add(f"{x:,}")
    walk(data)
    return nums


prose_numbers = extract_numbers(memo)
source_numbers = flatten_source_numbers(source_data)

# Match: prose number appears in source (or is derived — like a percentage or ratio)
verifiable = prose_numbers & source_numbers
# For unverifiable numbers, they're either percentages (derived) or fabrications
unverified = prose_numbers - source_numbers - {n for n in prose_numbers if '%' in n or n in {'1','2','3','4','5'}}

print(f"Numbers in memo prose: {len(prose_numbers)}")
print(f"  Verifiable against source: {len(verifiable)}")
print(f"  Percentages (derived, OK): {len({n for n in prose_numbers if '%' in n})}")
print(f"  Unverified (potentially hallucinated): {len(unverified)}")

if unverified:
    print(f"\n⚠  Unverified numbers to review:")
    for u in list(unverified)[:10]:
        print(f"    - {u}")

## Step 6 — Progressive compression

In [ ]:
def compress(text, audience, word_limit):
    prompt = f'''Rewrite this for {audience}. Hard limit: {word_limit} words.

Rules:
- Preserve numbers exactly
- Drop supporting details, keep top claims
- No adjectives without data (e.g., 'strong growth' → '18% growth')

Source:
{text}'''
    return llm.invoke(prompt).content.strip()


exec_summary = compress(memo, "CFO", 500)
board_bullets = compress(exec_summary, "board pre-read (bullet points)", 50)

print("═══ EXECUTIVE SUMMARY (~500 words) ═══")
print(exec_summary[:1500])
print(f"\n[Actual length: {len(exec_summary.split())} words]")

print("\n═══ BOARD BULLETS (~50 words) ═══")
print(board_bullets)
print(f"\n[Actual length: {len(board_bullets.split())} words]")

## Step 7 — Save all outputs

In [ ]:
outputs = Path("/tmp/day7_outputs")
outputs.mkdir(exist_ok=True)

(outputs / "full_memo.md").write_text(memo)
(outputs / "exec_summary.md").write_text(exec_summary)
(outputs / "board_bullets.md").write_text(board_bullets)

print(f"✓ Saved 3 report formats to {outputs}/")
for p in outputs.glob("*.md"):
    print(f"  {p.name}: {len(p.read_text().split())} words")

# Chart is at /tmp/day7_top_chart.png
if chart_path.exists():
    print(f"  {chart_path.name}: {chart_path.stat().st_size} bytes")

## What you learned

1. **Draft with structured sections** — each section rendered from insights, not from LLM in one shot
2. **LLM writes code, you execute** — deterministic charts, editable, versionable
3. **Consistency verification** — extract numbers from prose, verify against source
4. **Progressive compression** — same content, three audience-tuned formats
5. **`matplotlib.use('Agg')`** — headless backend for non-interactive execution

## Production notes

- Sandbox the `exec()` call in production (restricted globals, subprocess with timeout)
- Add HITL review gate before publishing (same interrupt_before pattern from Day 6)
- Log every chart generation prompt + code for audit
- Cache generated charts by (insight hash + data hash) — regenerate only when data changes

## Day 7 complete

You've built the full report pipeline: insights → memo → charts → verification → compression.

Day 8: end-to-end orchestration + capstone.